# 01 — Python untuk analisis

Bukan kursus Python umum: hanya bagian yang dipakai delapan modul berikutnya. Kamus (*dict*) sebagai
konfigurasi, DataFrame pandas, lima kata kerja utama (baca, saring, kelompokkan, agregasi, gabung), fungsi,
dan grafik untuk memeriksa pekerjaan.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## 1. Dict sebagai konfigurasi
Satu objek memuat semua parameter kota. Diteruskan ke setiap fungsi, disimpan sebagai JSON — pindah kota
berarti mengganti satu objek, bukan puluhan variabel.

In [ ]:
CITY = json.loads((Path("..") / os.environ.get("KOTA_CONFIG", "city_config.json")).read_text())
print(f"Kota aktif: {CITY['name']}, {CITY['province']} — EPSG:{CITY['epsg_utm']}")
print("Kunci:", list(CITY))

## 2. Membaca dan memeriksa DataFrame

In [ ]:
RAW = Path("../data/raw") / CITY["slug"]
kel = pd.read_csv(RAW / "kelurahan.csv")
kec = pd.read_csv(RAW / "kecamatan.csv")
print(kel.shape, kec.shape)
kel.dtypes

In [ ]:
kel.describe().round(1)

## 3. Kolom turunan, menyaring, mengurutkan

In [ ]:
kel["bangunan_per_ha"] = kel["n_bangunan"] / kel["luas_ha"]
padat = kel[(kel["bangunan_per_ha"] > 40) & (kel["luas_ha"] < 50)]
padat.sort_values("bangunan_per_ha", ascending=False).round(1)

## 4. Fungsi: tulis sekali, pakai berulang
Tulis fungsi ketika Anda mulai menyalin-tempel kode yang sama untuk kedua kalinya.

In [ ]:
def kelas_kepadatan(bph):
    """Kelas kepadatan bangunan (bangunan per hektar) — ambang untuk latihan."""
    if bph >= 35:
        return "tinggi"
    if bph >= 15:
        return "sedang"
    return "rendah"


kel["kelas"] = kel["bangunan_per_ha"].apply(kelas_kepadatan)
kel["kelas"].value_counts()

## 5. Kelompokkan dan agregasi

In [ ]:
per_kec = kel.groupby("kecamatan").agg(n_kelurahan=("kelurahan", "count"), n_bangunan=("n_bangunan", "sum"),
                                       luas_kel_ha=("luas_ha", "sum"))
per_kec.head()

## 6. Menggabungkan tabel
Luas kecamatan ada di tabel lain. `merge` pada kolom kunci — dan periksa selalu jumlah baris setelahnya.

In [ ]:
gab = kec.merge(per_kec, left_on="kecamatan", right_index=True, how="left", validate="one_to_one")
gab["bangunan_per_km2"] = (gab["n_bangunan"] / gab["luas_km2"]).round(0)
gab["selisih_luas_%"] = (100 * (gab["luas_kel_ha"] / 100 - gab["luas_km2"]) / gab["luas_km2"]).round(2)
print("baris:", len(kec), "→", len(gab), "| kecamatan tanpa pasangan:", gab["n_bangunan"].isna().sum())
gab.sort_values("bangunan_per_km2", ascending=False)

`selisih_luas_%` memeriksa konsistensi: jumlah luas kelurahan seharusnya sama dengan luas kecamatannya.

## 7. Grafik untuk memeriksa pekerjaan

In [ ]:
ax = gab.sort_values("bangunan_per_km2").plot.barh(x="kecamatan", y="bangunan_per_km2", color="#2A7B9B",
                                                   legend=False, figsize=(8, 6))
ax.set_xlabel("bangunan per km²")
ax.set_title(f"Kepadatan bangunan per kecamatan — {CITY['name']}")
plt.tight_layout()
Path("../output", CITY["slug"]).mkdir(parents=True, exist_ok=True)
plt.savefig(Path("../output", CITY["slug"], "kepadatan_kecamatan.png"), dpi=150)